# 01 - Data Exploration & Fetching

This notebook covers the foundation: fetching competitions, teams, and building the initial player dataset.

**Timeline**: Approximately 10-15 minutes for initial data fetch (depending on API response times).

## Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import all required modules
from src.config import KNOWN_COMPETITIONS, ACTIVE_COMPETITIONS
from src.ingestion import DataFetcher, WyscoutAPIClient
from src.utils import DataManager

import pandas as pd
import numpy as np
from pprint import pprint

print("✓ All modules imported successfully")
print(f"✓ Active competitions: {list(ACTIVE_COMPETITIONS.keys())}")


## Step 1: Fetch All Teams from Competitions

In [ ]:
# Initialize the data fetcher
fetcher = DataFetcher()

# Fetch competitions and their teams (excluding Primavera)
print("Fetching competitions and teams (excluding Primavera 1)...")
teams_by_comp = fetcher.fetch_competitions_and_teams(exclude_primavera=True)

print(f"\n✓ Fetched {len(teams_by_comp)} competitions:")
for comp_name, teams_list in teams_by_comp.items():
    print(f"  - {comp_name}: {len(teams_list)} teams")

# Save teams data for use in other scripts
manager = DataManager()
manager.save_json(fetcher.team_id_to_name, 'data/cache/team_id_to_name.json')
print("\n✓ Team mapping saved to cache")


## Step 2: Explore a Specific Team

In [ ]:
# Get first team from Serie A as an example
if teams_by_comp and 'Serie A' in teams_by_comp:
    first_team = teams_by_comp['Serie A'][0]
    team_wy_id = first_team['wyId']
    team_name = first_team['name']
    
    print(f"Fetching squad for: {team_name}")
    squad = fetcher.fetch_squad_for_team(team_wy_id)
    
    # Create DataFrame
    squad_df = pd.DataFrame(squad)
    print(f"\n✓ Squad for {team_name}:")
    print(squad_df.head(10))
else:
    print("No teams found")


## Step 3: Fetch Player Details

In [ ]:
# Get details for the first player in the squad
if squad and len(squad) > 0:
    first_player_wy_id = squad[0]['wyId']
    first_player_name = squad[0]['name']
    
    print(f"Fetching details for: {first_player_name}")
    player_details = fetcher.fetch_player_details(first_player_wy_id)
    
    if player_details:
        print(f"\n✓ Player Details:")
        pprint(player_details)
    else:
        print("Could not fetch player details")
else:
    print("No squad data available")


## Step 4: Fetch Complete Player Dataset (Optional - 20-30 minutes)

In [ ]:
# WARNING: This fetches ALL players from Serie A, B, C - can take 20-30 minutes
# Uncomment to run

# fetcher_all = DataFetcher()
# print("Fetching all players from Serie A, B, C (this will take ~20-30 minutes)...")
# all_players = fetcher_all.fetch_all_players_from_competitions(
#     competitions=['Serie A', 'Serie B', 'Serie C']
# )
# 
# players_df = pd.DataFrame(all_players)
# 
# # Save for next stage
# manager.save_pickle(players_df, 'data/processed/players_profiles.pkl')
# print(f"\n✓ Saved {len(players_df)} players to data/processed/players_profiles.pkl")
# print(players_df.head())


## Summary

This notebook demonstrates:
- ✓ Setting up the DataFetcher module
- ✓ Fetching competitions and teams
- ✓ Exploring team squads
- ✓ Retrieving player details
- ✓ (Optional) Bulk fetching all players

**Next Step**: Run `02_player_analysis.ipynb` to analyze player careers and enrich data.
